In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/yelp-dataset/Dataset_User_Agreement.pdf
/kaggle/input/yelp-dataset/yelp_academic_dataset_review.json
/kaggle/input/yelp-dataset/yelp_academic_dataset_checkin.json
/kaggle/input/yelp-dataset/yelp_academic_dataset_business.json
/kaggle/input/yelp-dataset/yelp_academic_dataset_tip.json
/kaggle/input/yelp-dataset/yelp_academic_dataset_user.json


In [5]:
import pandas as pd
import json

# Load JSON into a DataFrame
def load_json_file(filepath):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

business_df = load_json_file('/kaggle/input/yelp-dataset/yelp_academic_dataset_business.json')
print(f"Loaded {len(business_df)} business records")
business_df.head()

Loaded 150346 business records


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [6]:
# Show basic info
print("BUSINESS DATA STRUCTURE:")
print(business_df.info())

# Show column names
print("\nCOLUMNS AVAILABLE:")
print(business_df.columns.tolist())

# Check for nested JSON columns
print("\nNESTED COLUMNS INSPECTION:")
print("Attributes type:", type(business_df['attributes'].iloc[0]))
print("Hours type:", type(business_df['hours'].iloc[0]))

BUSINESS DATA STRUCTURE:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   business_id   150346 non-null  object 
 1   name          150346 non-null  object 
 2   address       150346 non-null  object 
 3   city          150346 non-null  object 
 4   state         150346 non-null  object 
 5   postal_code   150346 non-null  object 
 6   latitude      150346 non-null  float64
 7   longitude     150346 non-null  float64
 8   stars         150346 non-null  float64
 9   review_count  150346 non-null  int64  
 10  is_open       150346 non-null  int64  
 11  attributes    136602 non-null  object 
 12  categories    150243 non-null  object 
 13  hours         127123 non-null  object 
dtypes: float64(3), int64(2), object(9)
memory usage: 16.1+ MB
None

COLUMNS AVAILABLE:
['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'lati

In [7]:
# Check for nulls
business_df.isnull().sum()

business_id         0
name                0
address             0
city                0
state               0
postal_code         0
latitude            0
longitude           0
stars               0
review_count        0
is_open             0
attributes      13744
categories        103
hours           23223
dtype: int64

In [8]:
# Numeric stats
business_df.describe()

,latitude,longitude,stars,review_count,is_open
count,150346.000000,150346.000000,150346.000000,150346.000000,150346.00000
mean,36.671150,-89.357339,3.596724,44.866561,0.79615
std,5.872759,14.918502,0.974421,121.120136,0.40286
min,27.555127,-120.095137,1.000000,5.000000,0.00000
25%,32.187293,-90.357810,3.000000,8.000000,1.00000
50%,38.777413,-86.121179,3.500000,15.000000,1.00000
75%,39.954036,-75.421542,4.500000,37.000000,1.00000
max,53.679197,-73.200457,5.000000,7568.000000,1.00000


In [9]:
import pandas as pd
import json
import numpy as np

# Function to safely convert string to dict
def safe_json_loads(x):
    if pd.isna(x) or x == '':
        return {}
    try:
        return json.loads(x.replace("'", "\"")) if isinstance(x, str) else x
    except:
        return {}

# Clean the business data
def clean_business_data(df):
    # Convert string representations of dictionaries to actual dicts
    df['attributes'] = df['attributes'].apply(safe_json_loads)
    df['hours'] = df['hours'].apply(safe_json_loads)
    
    # Convert categories from string to list
    df['categories'] = df['categories'].str.split(', ').fillna('').apply(lambda x: [c.strip() for c in x] if x else [])
    
    return df

business_df = clean_business_data(business_df)

In [10]:
# Expand attributes into separate columns
def expand_attributes(df):
    # Get all unique attribute keys
    all_attrs = set()
    for attrs in df['attributes']:
        all_attrs.update(attrs.keys())
    
    # Create columns for each attribute
    for attr in all_attrs:
        df[f"attr_{attr}"] = df['attributes'].apply(lambda x: x.get(attr, np.nan))
    
    # Drop the original attributes column
    df = df.drop('attributes', axis=1)
    return df

business_df = expand_attributes(business_df)

In [11]:
# Expand hours into separate columns
def expand_hours(df):
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    for day in days:
        df[f"hours_{day}"] = df['hours'].apply(lambda x: x.get(day, np.nan))
    
    # Optionally split hours into open/close times
    for day in days:
        df[[f"hours_{day}_open", f"hours_{day}_close"]] = df[f"hours_{day}"].str.split('-', expand=True)
        df = df.drop(f"hours_{day}", axis=1)
    
    df = df.drop('hours', axis=1)
    return df

business_df = expand_hours(business_df)

In [9]:
# # Option 1: Keep as comma-separated string (simpler for SQL)
# business_df['categories'] = business_df['categories'].apply(lambda x: ', '.join(x) if x else '')

# # Option 2: Create a separate categories table (better normalized)
# # This would be a second table with business_id and category columns

In [12]:
# Convert boolean-like strings to actual booleans
bool_cols = [col for col in business_df.columns if col.startswith('attr_') and business_df[col].isin(['True', 'False']).any()]
for col in bool_cols:
    business_df[col] = business_df[col].map({'True': True, 'False': False})

# Convert numeric columns
numeric_cols = ['latitude', 'longitude', 'stars', 'review_count', 'is_open']
business_df[numeric_cols] = business_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Clean text columns
text_cols = ['city', 'state', 'postal_code']
business_df[text_cols] = business_df[text_cols].apply(lambda x: x.str.strip())

In [13]:
print("\nData Summary:")
print(business_df.info())

# print("\nDescriptive Statistics:")
# print(business_df.describe(include='all'))


Data Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 65 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   business_id                      150346 non-null  object 
 1   name                             150346 non-null  object 
 2   address                          150346 non-null  object 
 3   city                             150346 non-null  object 
 4   state                            150346 non-null  object 
 5   postal_code                      150346 non-null  object 
 6   latitude                         150346 non-null  float64
 7   longitude                        150346 non-null  float64
 8   stars                            150346 non-null  float64
 9   review_count                     150346 non-null  int64  
 10  is_open                          150346 non-null  int64  
 11  categories                       150346 non-null  

In [14]:
business_df.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,hours_Wednesday_open,hours_Wednesday_close,hours_Thursday_open,hours_Thursday_close,hours_Friday_open,hours_Friday_close,hours_Saturday_open,hours_Saturday_close,hours_Sunday_open,hours_Sunday_close
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,...,8:0,18:30,8:0,18:30,8:0,18:30,8:0,14:0,NaN,NaN
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,...,8:0,22:0,8:0,22:0,8:0,23:0,8:0,23:0,8:0,22:0
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,...,7:0,20:0,7:0,20:0,7:0,21:0,7:0,21:0,7:0,21:0
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,...,14:0,22:0,16:0,22:0,12:0,22:0,12:0,22:0,12:0,18:0


In [15]:
# Display just the categories column
print("=== Categories Column (First 10 Rows) ===")
print(business_df['categories'].head(10))

=== Categories Column (First 10 Rows) ===
0    [Doctors, Traditional Chinese Medicine, Naturo...
1    [Shipping Centers, Local Services, Notaries, M...
2    [Department Stores, Shopping, Fashion, Home & ...
3    [Restaurants, Food, Bubble Tea, Coffee & Tea, ...
4                          [Brewpubs, Breweries, Food]
5    [Burgers, Fast Food, Sandwiches, Food, Ice Cre...
6    [Sporting Goods, Fashion, Shoe Stores, Shoppin...
7                [Synagogues, Religious Organizations]
8    [Pubs, Restaurants, Italian, Bars, American (T...
9    [Ice Cream & Frozen Yogurt, Fast Food, Burgers...
Name: categories, dtype: object


In [16]:
# Create cleaned version excluding only the categories column
business_clean = business_df.drop(columns=['categories']).copy()

# Verify the result
print("Columns in business_clean:", business_clean.columns.tolist())
business_clean.head()

Columns in business_clean: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attr_RestaurantsGoodForGroups', 'attr_BusinessAcceptsCreditCards', 'attr_HappyHour', 'attr_DogsAllowed', 'attr_BestNights', 'attr_RestaurantsPriceRange2', 'attr_BusinessParking', 'attr_RestaurantsAttire', 'attr_BusinessAcceptsBitcoin', 'attr_Smoking', 'attr_WheelchairAccessible', 'attr_NoiseLevel', 'attr_Alcohol', 'attr_GoodForKids', 'attr_GoodForMeal', 'attr_AgesAllowed', 'attr_DietaryRestrictions', 'attr_RestaurantsReservations', 'attr_OutdoorSeating', 'attr_Open24Hours', 'attr_ByAppointmentOnly', 'attr_BYOBCorkage', 'attr_DriveThru', 'attr_CoatCheck', 'attr_Caters', 'attr_BYOB', 'attr_GoodForDancing', 'attr_WiFi', 'attr_Corkage', 'attr_BikeParking', 'attr_RestaurantsTableService', 'attr_Music', 'attr_RestaurantsDelivery', 'attr_HairSpecializesIn', 'attr_AcceptsInsurance', 'attr_HasTV', 'attr_RestaurantsTakeOut', 'attr_Ambience', 

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,hours_Wednesday_open,hours_Wednesday_close,hours_Thursday_open,hours_Thursday_close,hours_Friday_open,hours_Friday_close,hours_Saturday_open,hours_Saturday_close,hours_Sunday_open,hours_Sunday_close
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,...,8:0,18:30,8:0,18:30,8:0,18:30,8:0,14:0,NaN,NaN
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,...,8:0,22:0,8:0,22:0,8:0,23:0,8:0,23:0,8:0,22:0
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,...,7:0,20:0,7:0,20:0,7:0,21:0,7:0,21:0,7:0,21:0
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,...,14:0,22:0,16:0,22:0,12:0,22:0,12:0,22:0,12:0,18:0


In [17]:
# Explode categories into a separate table
business_categories = business_df[['business_id', 'categories']].dropna().copy()
business_categories['categories'] = business_categories['categories'].str.split(', ')

# Explode into rows
business_categories = business_categories.explode('categories').reset_index(drop=True)
business_categories.rename(columns={'categories': 'category'}, inplace=True)

# Preview
business_categories.head()

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,business_id,category
0,Pns2l4eNsfO8kk83dixA6A,NaN
1,mpf3x-BjTdTEA3yCZrAYPw,NaN
2,tUFrWirKiKi_TAnsVWINQQ,NaN
3,MTSW4McQd7CbVtyjqoe9mw,NaN
4,mWMc6_wTdE0EUBKIGXDVfA,NaN


In [16]:
# def flatten_json_columns(df, column_names):
#     """Flatten nested JSON columns into separate columns"""
#     for col in column_names:
#         if col in df.columns:
#             # Handle null values
#             df[col] = df[col].apply(lambda x: x if isinstance(x, dict) else {})
#             flattened = json_normalize(df[col])
#             flattened.columns = [f"{col}_{subcol}" for subcol in flattened.columns]
#             df = pd.concat([df.drop(col, axis=1), flattened], axis=1)
#     return df

# # Flatten attributes and hours
# business_df = flatten_json_columns(business_df, ['attributes', 'hours'])
# print("\nFLATTENED COLUMNS:")
# print(business_df.columns.tolist())

In [17]:
# # Handle categories (which is a list of strings)
# business_df['categories'] = business_df['categories'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')

# # Convert boolean columns to 1/0
# bool_cols = business_df.select_dtypes(include=['bool']).columns
# business_df[bool_cols] = business_df[bool_cols].astype(int)

# # Convert all text columns to string and limit length
# text_cols = business_df.select_dtypes(include=['object']).columns
# business_df[text_cols] = business_df[text_cols].astype(str).apply(lambda x: x.str[:255])

# # Show cleaned data
# business_df.head()

In [18]:
import pandas as pd
import json

# Load check-in data
def load_json_file(filepath):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

checkin_df = load_json_file('/kaggle/input/yelp-dataset/yelp_academic_dataset_checkin.json')
print(f"Loaded {len(checkin_df)} check-in records")
checkin_df.head()

Loaded 131930 check-in records


,business_id,date
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020..."
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011..."
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22"
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012..."
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014..."


In [19]:
# Info and null values
checkin_df.info()
checkin_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131930 entries, 0 to 131929
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   business_id  131930 non-null  object
 1   date         131930 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB


business_id    0
date           0
dtype: int64

In [20]:
# Process check-in data
checkin_clean = checkin_df.dropna(subset=['date'])  # Remove null dates

# Split comma-separated dates and create one row per check-in
checkin_clean = (checkin_clean.assign(date=checkin_clean['date'].str.split(', '))
                 .explode('date')
                 .reset_index(drop=True))

# Convert to datetime and clean up
checkin_clean['checkin_time'] = pd.to_datetime(checkin_clean['date'])
checkin_clean = checkin_clean.drop(columns=['date'])

print("Cleaned Check-in Data:")
checkin_clean.head()

Cleaned Check-in Data:


,business_id,checkin_time
0,---kPU91CF4Lq2-WlRu9Lw,2020-03-13 21:10:56
1,---kPU91CF4Lq2-WlRu9Lw,2020-06-02 22:18:06
2,---kPU91CF4Lq2-WlRu9Lw,2020-07-24 22:42:27
3,---kPU91CF4Lq2-WlRu9Lw,2020-10-24 21:36:13
4,---kPU91CF4Lq2-WlRu9Lw,2020-12-09 21:23:33


In [21]:
# Initialize an empty list to store the chunks
chunks = []

# Load the review dataset in chunks
for chunk in pd.read_json('/kaggle/input/yelp-dataset/yelp_academic_dataset_review.json', 
                         lines=True, 
                         chunksize=100000):
    chunks.append(chunk)

# Concatenate all chunks into a single DataFrame
review_df = pd.concat(chunks, ignore_index=True)

# Remove duplicate columns if any
review_df = review_df.loc[:, ~review_df.columns.duplicated()]

print(f"Loaded {len(review_df)} review records")
review_df.head()

Loaded 6990280 review records


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


In [22]:
# Info and null values
review_df.info()
review_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6990280 entries, 0 to 6990279
Data columns (total 9 columns):
 #   Column       Dtype         
---  ------       -----         
 0   review_id    object        
 1   user_id      object        
 2   business_id  object        
 3   stars        int64         
 4   useful       int64         
 5   funny        int64         
 6   cool         int64         
 7   text         object        
 8   date         datetime64[ns]
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 480.0+ MB


review_id      0
user_id        0
business_id    0
stars          0
useful         0
funny          0
cool           0
text           0
date           0
dtype: int64

In [23]:
import pandas as pd
import json

# Load tip data
def load_json_file(filepath):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

tip_df = load_json_file('/kaggle/input/yelp-dataset/yelp_academic_dataset_tip.json')
print(f"Loaded {len(tip_df)} review records")
tip_df.head()

Loaded 908915 review records


,user_id,business_id,text,date,compliment_count
0,AGNUgVwnZUey3gcPCJ76iw,3uLgwr0qeCNMjKenHJwPGQ,Avengers time with the ladies.,2012-05-18 02:17:21,0
1,NBN4MgHP9D3cw--SnauTkA,QoezRbYQncpRqyrLH6Iqjg,They have lots of good deserts and tasty cuban...,2013-02-05 18:35:10,0
2,-copOvldyKh1qr-vzkDEvw,MYoRNLb5chwjQe3c_k37Gg,It's open even when you think it isn't,2013-08-18 00:56:08,0
3,FjMQVZjSqY8syIO-53KFKw,hV-bABTK-glh5wj31ps_Jw,Very decent fried chicken,2017-06-27 23:05:38,0
4,ld0AperBXk1h6UbqmM80zw,_uN0OudeJ3Zl_tf6nxg5ww,Appetizers.. platter special for lunch,2012-10-06 19:43:09,0


In [24]:
# Info and null values
tip_df.info()
tip_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 908915 entries, 0 to 908914
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   user_id           908915 non-null  object
 1   business_id       908915 non-null  object
 2   text              908915 non-null  object
 3   date              908915 non-null  object
 4   compliment_count  908915 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 34.7+ MB


user_id             0
business_id         0
text                0
date                0
compliment_count    0
dtype: int64

In [25]:
# Convert date to datetime
tip_df['date'] = pd.to_datetime(tip_df['date'])

# Final cleaned version
tip_clean = tip_df[['user_id', 'business_id', 'text', 'date', 'compliment_count']].copy()

# Preview
tip_clean.head()

,user_id,business_id,text,date,compliment_count
0,AGNUgVwnZUey3gcPCJ76iw,3uLgwr0qeCNMjKenHJwPGQ,Avengers time with the ladies.,2012-05-18 02:17:21,0
1,NBN4MgHP9D3cw--SnauTkA,QoezRbYQncpRqyrLH6Iqjg,They have lots of good deserts and tasty cuban...,2013-02-05 18:35:10,0
2,-copOvldyKh1qr-vzkDEvw,MYoRNLb5chwjQe3c_k37Gg,It's open even when you think it isn't,2013-08-18 00:56:08,0
3,FjMQVZjSqY8syIO-53KFKw,hV-bABTK-glh5wj31ps_Jw,Very decent fried chicken,2017-06-27 23:05:38,0
4,ld0AperBXk1h6UbqmM80zw,_uN0OudeJ3Zl_tf6nxg5ww,Appetizers.. platter special for lunch,2012-10-06 19:43:09,0


In [26]:
# User Data
import pandas as pd
import json

def load_json_file(filepath):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# Load user data
user_df = load_json_file('/kaggle/input/yelp-dataset/yelp_academic_dataset_user.json')
print(f"Loaded {len(user_df)} user records")
user_df.head()

Loaded 1987897 user records


,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,...,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,...,1,0,0,0,1,1,0,0,0,0


In [27]:
# Info and null values
user_df.info()
user_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1987897 entries, 0 to 1987896
Data columns (total 22 columns):
 #   Column              Dtype  
---  ------              -----  
 0   user_id             object 
 1   name                object 
 2   review_count        int64  
 3   yelping_since       object 
 4   useful              int64  
 5   funny               int64  
 6   cool                int64  
 7   elite               object 
 8   friends             object 
 9   fans                int64  
 10  average_stars       float64
 11  compliment_hot      int64  
 12  compliment_more     int64  
 13  compliment_profile  int64  
 14  compliment_cute     int64  
 15  compliment_list     int64  
 16  compliment_note     int64  
 17  compliment_plain    int64  
 18  compliment_cool     int64  
 19  compliment_funny    int64  
 20  compliment_writer   int64  
 21  compliment_photos   int64  
dtypes: float64(1), int64(16), object(5)
memory usage: 333.7+ MB


user_id               0
name                  0
review_count          0
yelping_since         0
useful                0
funny                 0
cool                  0
elite                 0
friends               0
fans                  0
average_stars         0
compliment_hot        0
compliment_more       0
compliment_profile    0
compliment_cute       0
compliment_list       0
compliment_note       0
compliment_plain      0
compliment_cool       0
compliment_funny      0
compliment_writer     0
compliment_photos     0
dtype: int64

In [28]:
# Convert 'yelping_since' to datetime
user_df['yelping_since'] = pd.to_datetime(user_df['yelping_since'])

In [30]:
# Install unixODBC
!apt-get update
!apt-get install -y unixodbc unixodbc-dev

# Install pyodbc
!pip install pyodbc

# Install ODBC Driver 17 for SQL Server
!curl https://packages.microsoft.com/keys/microsoft.asc | apt-key add -
!curl https://packages.microsoft.com/config/ubuntu/$(lsb_release -rs)/prod.list | tee /etc/apt/sources.list.d/mssql-release.list
!apt-get update
!ACCEPT_EULA=Y apt-get install -y msodbcsql17

 

import urllib
from sqlalchemy import create_engine
from sqlalchemy.sql import text

username = 'group9studentuser'  # Replace with your Azure SQL admin username
password = 'PasswordGroup9'        # Replace with your Azure SQL password
server = 'team9azureuser.database.windows.net'  # Replace with your Azure SQL server name
database = 'StudemtDB_9'  # Replace with your Azure SQL database

# Define connection parameters
driver = "ODBC Driver 17 for SQL Server"  # Ensure this driver is installed
 
# Build the ODBC connection string
odbc_str = f"DRIVER={{{driver}}};SERVER={server};PORT=1433;DATABASE={database};UID={username};PWD={password}"
 
# Encode the connection string for SQLAlchemy
connection_url = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(odbc_str)}"
 
# Create SQLAlchemy engine
engine = create_engine(connection_url)
 

with engine.connect() as connection:
    result = connection.execute(text("SELECT GETDATE();"))
    for row in result:
        print("Connected! Server date and time:", row[0])

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease         
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease                          
Hit:4 https://packages.microsoft.com/ubuntu/22.04/prod jammy InRelease                              
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                                              
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                                          
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                   
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: https://pack

In [29]:
!pip install pyodbc sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.0/336.0 kB 5.9 MB/s eta 0:00:00a 0:00:01


In [31]:
!curl ifconfig.me

34.75.232.187

In [32]:
def upload_dataframe_in_chunks(df, table_name, chunk_size=20000):
    total_rows = len(df)
    print(f"Uploading {total_rows} rows to '{table_name}' in chunks of {chunk_size}...")

    for start in range(0, total_rows, chunk_size):
        end = start + chunk_size
        chunk = df.iloc[start:end]

        if_exists_mode = 'replace' if start == 0 else 'append'

        try:
            chunk.to_sql(
                name=table_name,
                con=engine,
                if_exists=if_exists_mode,
                index=False
            )
            print(f"✅ Rows {start} to {end} uploaded")
        except Exception as e:
            print(f"❌ Error uploading rows {start} to {end}: {e}")
            break

In [36]:
upload_dataframe_in_chunks(business_clean, 'business_df')

Uploading 150346 rows to 'business_df' in chunks of 20000...
✅ Rows 0 to 20000 uploaded
✅ Rows 20000 to 40000 uploaded
✅ Rows 40000 to 60000 uploaded
✅ Rows 60000 to 80000 uploaded
✅ Rows 80000 to 100000 uploaded
✅ Rows 100000 to 120000 uploaded
✅ Rows 120000 to 140000 uploaded
✅ Rows 140000 to 160000 uploaded


In [33]:
upload_dataframe_in_chunks(tip_clean, 'tip_df')

Uploading 908915 rows to 'tip_df' in chunks of 20000...
✅ Rows 0 to 20000 uploaded
✅ Rows 20000 to 40000 uploaded
✅ Rows 40000 to 60000 uploaded
✅ Rows 60000 to 80000 uploaded
✅ Rows 80000 to 100000 uploaded
✅ Rows 100000 to 120000 uploaded
✅ Rows 120000 to 140000 uploaded
✅ Rows 140000 to 160000 uploaded
✅ Rows 160000 to 180000 uploaded
✅ Rows 180000 to 200000 uploaded
✅ Rows 200000 to 220000 uploaded
✅ Rows 220000 to 240000 uploaded
✅ Rows 240000 to 260000 uploaded
✅ Rows 260000 to 280000 uploaded
✅ Rows 280000 to 300000 uploaded
✅ Rows 300000 to 320000 uploaded
✅ Rows 320000 to 340000 uploaded
✅ Rows 340000 to 360000 uploaded
✅ Rows 360000 to 380000 uploaded
✅ Rows 380000 to 400000 uploaded
✅ Rows 400000 to 420000 uploaded
✅ Rows 420000 to 440000 uploaded
✅ Rows 440000 to 460000 uploaded
✅ Rows 460000 to 480000 uploaded
✅ Rows 480000 to 500000 uploaded
✅ Rows 500000 to 520000 uploaded
✅ Rows 520000 to 540000 uploaded
✅ Rows 540000 to 560000 uploaded
✅ Rows 560000 to 580000 uploaded


In [34]:
upload_dataframe_in_chunks(checkin_clean, 'checkin_df')

Uploading 13356875 rows to 'checkin_df' in chunks of 20000...
✅ Rows 0 to 20000 uploaded
✅ Rows 20000 to 40000 uploaded
✅ Rows 40000 to 60000 uploaded
✅ Rows 60000 to 80000 uploaded
✅ Rows 80000 to 100000 uploaded
✅ Rows 100000 to 120000 uploaded
✅ Rows 120000 to 140000 uploaded
✅ Rows 140000 to 160000 uploaded
✅ Rows 160000 to 180000 uploaded
✅ Rows 180000 to 200000 uploaded
✅ Rows 200000 to 220000 uploaded
✅ Rows 220000 to 240000 uploaded
✅ Rows 240000 to 260000 uploaded
✅ Rows 260000 to 280000 uploaded
✅ Rows 280000 to 300000 uploaded
✅ Rows 300000 to 320000 uploaded
✅ Rows 320000 to 340000 uploaded
✅ Rows 340000 to 360000 uploaded
✅ Rows 360000 to 380000 uploaded
✅ Rows 380000 to 400000 uploaded
✅ Rows 400000 to 420000 uploaded
✅ Rows 420000 to 440000 uploaded
✅ Rows 440000 to 460000 uploaded
✅ Rows 460000 to 480000 uploaded
✅ Rows 480000 to 500000 uploaded
✅ Rows 500000 to 520000 uploaded
✅ Rows 520000 to 540000 uploaded
✅ Rows 540000 to 560000 uploaded
✅ Rows 560000 to 580000 upl

In [35]:
upload_dataframe_in_chunks(user_df, 'user_df')

Uploading 1987897 rows to 'user_df' in chunks of 20000...
✅ Rows 0 to 20000 uploaded
✅ Rows 20000 to 40000 uploaded
✅ Rows 40000 to 60000 uploaded
✅ Rows 60000 to 80000 uploaded
✅ Rows 80000 to 100000 uploaded
✅ Rows 100000 to 120000 uploaded
✅ Rows 120000 to 140000 uploaded
✅ Rows 140000 to 160000 uploaded
✅ Rows 160000 to 180000 uploaded
✅ Rows 180000 to 200000 uploaded
✅ Rows 200000 to 220000 uploaded
✅ Rows 220000 to 240000 uploaded
✅ Rows 240000 to 260000 uploaded
✅ Rows 260000 to 280000 uploaded
✅ Rows 280000 to 300000 uploaded
✅ Rows 300000 to 320000 uploaded
✅ Rows 320000 to 340000 uploaded
✅ Rows 340000 to 360000 uploaded
✅ Rows 360000 to 380000 uploaded
✅ Rows 380000 to 400000 uploaded
✅ Rows 400000 to 420000 uploaded
✅ Rows 420000 to 440000 uploaded
✅ Rows 440000 to 460000 uploaded
✅ Rows 460000 to 480000 uploaded
✅ Rows 480000 to 500000 uploaded
✅ Rows 500000 to 520000 uploaded
✅ Rows 520000 to 540000 uploaded
✅ Rows 540000 to 560000 uploaded
✅ Rows 560000 to 580000 uploade

In [36]:
upload_dataframe_in_chunks(review_df, 'review_df')

Uploading 6990280 rows to 'review_df' in chunks of 20000...
✅ Rows 0 to 20000 uploaded
✅ Rows 20000 to 40000 uploaded
✅ Rows 40000 to 60000 uploaded
✅ Rows 60000 to 80000 uploaded
✅ Rows 80000 to 100000 uploaded
✅ Rows 100000 to 120000 uploaded
✅ Rows 120000 to 140000 uploaded
✅ Rows 140000 to 160000 uploaded
✅ Rows 160000 to 180000 uploaded
✅ Rows 180000 to 200000 uploaded
✅ Rows 200000 to 220000 uploaded
✅ Rows 220000 to 240000 uploaded
✅ Rows 240000 to 260000 uploaded
✅ Rows 260000 to 280000 uploaded
✅ Rows 280000 to 300000 uploaded
✅ Rows 300000 to 320000 uploaded
✅ Rows 320000 to 340000 uploaded
✅ Rows 340000 to 360000 uploaded
✅ Rows 360000 to 380000 uploaded
✅ Rows 380000 to 400000 uploaded
✅ Rows 400000 to 420000 uploaded
✅ Rows 420000 to 440000 uploaded
✅ Rows 440000 to 460000 uploaded
✅ Rows 460000 to 480000 uploaded
✅ Rows 480000 to 500000 uploaded
✅ Rows 500000 to 520000 uploaded
✅ Rows 520000 to 540000 uploaded
✅ Rows 540000 to 560000 uploaded
✅ Rows 560000 to 580000 uploa

In [37]:
upload_dataframe_in_chunks(business_categories, 'business_categories')

Uploading 150346 rows to 'business_categories' in chunks of 20000...
✅ Rows 0 to 20000 uploaded
✅ Rows 20000 to 40000 uploaded
✅ Rows 40000 to 60000 uploaded
✅ Rows 60000 to 80000 uploaded
✅ Rows 80000 to 100000 uploaded
✅ Rows 100000 to 120000 uploaded
✅ Rows 120000 to 140000 uploaded
✅ Rows 140000 to 160000 uploaded
